# 🧭 13 · 指标体系与埋点设计

> 数据分析师的「基础设施」能力：**怎么从业务目标推导出指标体系，怎么定义每一个指标的口径，怎么用埋点把数据采上来**。
> 本节覆盖北极星指标、OSM/AARRR、指标树、指标字典、埋点规范与数据校验、数仓分层。

## 📋 本节要解决的问题

| # | 问题 | 工具/方法 |
|---|---|---|
| 1 | 怎么从业务目标推导指标体系？ | 北极星 → 一级 → 二级 → 过程指标 |
| 2 | OSM / AARRR 怎么用？ | 目标-策略-度量 / 用户生命周期 |
| 3 | 指标口径怎么定义与沉淀？ | 指标字典 |
| 4 | 埋点怎么设计？ | 事件 × 属性、公共属性 |
| 5 | 埋点数据怎么校验？ | 缺失/重复/异常值、交叉核对 |
| 6 | 数据仓库怎么分层？ | ODS / DWD / DWS / ADS |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 北极星指标与指标树

**北极星指标**：唯一能代表产品长期价值的指标（如 Spotify 的"收听时长"、Airbnb 的"过夜数"）。
从北极星向下拆成**一级指标 → 二级指标 → 过程指标**，形成可监控、可归因的树：

In [2]:
kpi_tree = {
    '北极星': '月订单量',
    '一级': ['新客订单量', '老客订单量'],
    '二级': ['新客流量', '新客转化率', '老客复购率', '老客客单价'],
    '过程': ['曝光→点击率', '点击→加购率', '加购→支付率', '优惠券核销率'],
}
for level, items in kpi_tree.items():
    print(f'{level}: ' + ' → '.join(items))

# 可视化指标树（简化示意）
fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
levels = list(kpi_tree.items())
y_pos = [0.9, 0.7, 0.45, 0.2]
for (lv, items), y in zip(levels, y_pos):
    ax.text(0.5, y, f'【{lv}】', ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#dbeafe', edgecolor='#2563eb'))
    for i, it in enumerate(items):
        x = 0.15 + 0.7 * i / max(len(items) - 1, 1)
        ax.plot([0.5, x], [y - 0.045, y - 0.12], color='#94a3b8', lw=1)
        ax.text(x, y - 0.14, it, ha='center', fontsize=10, rotation=15)
ax.set_title('指标树示意：北极星 → 一级 → 二级 → 过程指标')
plt.show()

北极星: 月 → 订 → 单 → 量
一级: 新客订单量 → 老客订单量
二级: 新客流量 → 新客转化率 → 老客复购率 → 老客客单价
过程: 曝光→点击率 → 点击→加购率 → 加购→支付率 → 优惠券核销率


C:\Users\24260\AppData\Local\Temp\ipykernel_25368\1025128318.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2️⃣ OSM 模型：目标 → 策略 → 度量

OSM（Objective-Strategy-Measurement）：先想**目标**，再定**策略**，最后选**度量**，避免"先看数据再找意义"。

In [3]:
osm = pd.DataFrame([
    ('提升新客首购率', '新人礼包 + 首单立减', '首购率 / 新人领取率 / 礼包成本'),
    ('提升复购',        '会员体系 + 复购券',  '复购率 / 会员渗透率 / LTV'),
    ('提升客单价',      '满减 + 组合推荐',    '客单价 / 连带率 / 优惠券 ROI'),
], columns=['目标(O)', '策略(S)', '度量(M)'])
print(osm.to_string(index=False))
print('\n➡️ 面试回答模板：「目标是 X → 策略是 Y → 用 Z 衡量效果」')

  目标(O)       策略(S)               度量(M)
提升新客首购率 新人礼包 + 首单立减  首购率 / 新人领取率 / 礼包成本
   提升复购  会员体系 + 复购券   复购率 / 会员渗透率 / LTV
  提升客单价   满减 + 组合推荐 客单价 / 连带率 / 优惠券 ROI

➡️ 面试回答模板：「目标是 X → 策略是 Y → 用 Z 衡量效果」


## 3️⃣ AARRR：用户生命周期指标

In [4]:
aarrr = pd.DataFrame([
    ('Acquisition 获取',     '新增用户数 / 获客成本 CAC / 渠道质量'),
    ('Activation 激活',      '激活率 / 完成核心动作用户占比'),
    ('Retention 留存',       '次日/7日/30日留存率（分渠道）'),
    ('Revenue 收入',         'ARPU / ARPPU / GMV / LTV'),
    ('Referral 推荐',        '分享率 / K因子 / 邀请转化率'),
], columns=['阶段', '代表指标'])
print(aarrr.to_string(index=False))
assert '次日/7日/30日留存率（分渠道）' in aarrr['代表指标'].values
print('\n✅ AARRR 五个阶段指标齐全')

            阶段                     代表指标
Acquisition 获取  新增用户数 / 获客成本 CAC / 渠道质量
 Activation 激活         激活率 / 完成核心动作用户占比
  Retention 留存        次日/7日/30日留存率（分渠道）
    Revenue 收入 ARPU / ARPPU / GMV / LTV
   Referral 推荐        分享率 / K因子 / 邀请转化率

✅ AARRR 五个阶段指标齐全


## 4️⃣ 指标字典：口径沉淀（防止"各说各话"）

In [5]:
metric_dict = pd.DataFrame([
    ('DAU', '每日活跃用户数', '去重 user_id', '日', '客户成功'),
    ('次日留存率', '当日新增用户中次日仍活跃的比例', '次日活跃∩当日新增 / 当日新增', '日', '增长'),
    ('GMV', '成交总额（实付口径）', 'SUM(支付金额)，不含退款', '日/月', '电商'),
    ('转化率', '完成目标行为/到达入口', '支付人数 / 进入详情页人数', '日', '电商'),
    ('ARPU', '每活跃用户收入', '收入 / DAU', '日', '商业化'),
], columns=['指标', '定义', '口径/公式', '粒度', '负责团队'])
print(metric_dict.to_string(index=False))
print('\n➡️ 指标字典解决：同名不同口径、口径变更无记录、跨团队对不上数')

   指标              定义            口径/公式  粒度 负责团队
  DAU         每日活跃用户数       去重 user_id   日 客户成功
次日留存率 当日新增用户中次日仍活跃的比例 次日活跃∩当日新增 / 当日新增   日   增长
  GMV      成交总额（实付口径）   SUM(支付金额)，不含退款 日/月   电商
  转化率     完成目标行为/到达入口   支付人数 / 进入详情页人数   日   电商
 ARPU         每活跃用户收入         收入 / DAU   日  商业化

➡️ 指标字典解决：同名不同口径、口径变更无记录、跨团队对不上数


## 5️⃣ 埋点设计：事件 × 属性

In [6]:
# 埋点规范核心字段
event_schema = {
    '公共属性': 'app_version / platform / user_id / device_id / timestamp / session_id / os',
    '事件属性': {
        'page_view':   'page_id / refer_page',
        'click':       'element_id / page_id / position',
        'add_to_cart': 'sku_id / price / from(列表页/详情页/推荐)',
        'pay':         'order_id / amount / pay_method / coupon',
    },
    '命名规范': '小写下划线：page_view / add_to_cart；动宾结构',
    '取数要求': '前后端同时埋点可交叉校验；关键漏斗全链路覆盖',
}
for k, v in event_schema.items():
    print(f'{k}:')
    if isinstance(v, dict):
        for e, attrs in v.items():
            print(f'    {e}: {attrs}')
    else:
        print(f'    {v}')

公共属性:
    app_version / platform / user_id / device_id / timestamp / session_id / os
事件属性:
    page_view: page_id / refer_page
    click: element_id / page_id / position
    add_to_cart: sku_id / price / from(列表页/详情页/推荐)
    pay: order_id / amount / pay_method / coupon
命名规范:
    小写下划线：page_view / add_to_cart；动宾结构
取数要求:
    前后端同时埋点可交叉校验；关键漏斗全链路覆盖


## 6️⃣ 埋点数据校验：模拟埋点日志检查

In [7]:
# 模拟一天的部分埋点日志（含缺失/重复/异常）
log = pd.DataFrame({
    'event': rng.choice(['page_view', 'click', 'add_to_cart', 'pay'], 400),
    'user_id': rng.integers(1, 100, 400),
    'ts': rng.integers(1700000000, 1700086400, 400),
})
log.loc[5, 'user_id'] = None        # 缺失
log.loc[10, 'event'] = 'PageView'   # 命名不规范（大小写）
log.loc[20, 'ts'] = 0               # 时间异常
dup = pd.concat([log, log.iloc[[3]]])  # 故意加一条重复

# 校验 1：缺失率
missing_rate = dup['user_id'].isna().mean()
# 校验 2：重复率
dup_rate = dup.duplicated().mean()
# 校验 3：事件命名规范率
bad_name = (~dup['event'].str.islower()).mean()
# 校验 4：时间合理率
ts_ok = ((dup['ts'] > 1700000000) & (dup['ts'] < 1700086400)).mean()

qc = pd.DataFrame({
    '检查项': ['缺失率', '重复率', '命名不规范率', '时间异常率'],
    '数值': [missing_rate, dup_rate, bad_name, 1 - ts_ok],
})
print(qc.round(4).to_string(index=False))
assert dup_rate > 0 and bad_name > 0
print('\n✅ 四类问题均可被自动校验捕获 → 输出埋点质量报告')

   检查项     数值
   缺失率 0.0025
   重复率 0.0025
命名不规范率 0.0025
 时间异常率 0.0025

✅ 四类问题均可被自动校验捕获 → 输出埋点质量报告


## 7️⃣ 数据仓库分层（业务视角）

In [8]:
dw_layers = pd.DataFrame([
    ('ODS 原始层',  '原样接入业务库/埋点日志，仅备份不加工'),
    ('DWD 明细层',  '清洗、去重、标准化，生成明细事实与维度表'),
    ('DWS 汇总层',  '按主题宽表化（用户/商品/订单日汇总），指标预计算'),
    ('ADS 应用层',  '面向报表/看板/算法的指标结果表'),
], columns=['分层', '定位'])
print(dw_layers.to_string(index=False))
print('\n➡️ 面试常见追问：为什么分四层？(解耦/复用/性能/权限) 分层越多越好吗？(数据延迟与维护成本)')

     分层                        定位
ODS 原始层       原样接入业务库/埋点日志，仅备份不加工
DWD 明细层      清洗、去重、标准化，生成明细事实与维度表
DWS 汇总层 按主题宽表化（用户/商品/订单日汇总），指标预计算
ADS 应用层          面向报表/看板/算法的指标结果表

➡️ 面试常见追问：为什么分四层？(解耦/复用/性能/权限) 分层越多越好吗？(数据延迟与维护成本)


## 8️⃣ 面试高频题（指标体系方向）

- **怎么给一个产品建指标体系**：北极星 → 一级/二级 → 过程；再补护栏指标（体验/成本）
- **北极星指标怎么选**：长期价值相关、可衡量、可被团队影响、不能"刷"（如 DAU 可被拉新刷）
- **指标口径冲突怎么处理**：指标字典 + 单一数据源（One Data）+ 变更记录
- **埋点质量怎么保证**：前后端双埋点、自动化校验（缺失/重复/异常/命名）、口径测试用例
- **数仓分几层**：ODS→DWD→DWS→ADS，讲清楚每层职责
- **看板怎么搭**：北极星在最顶、下钻维度一致、异常预警自动通知

## 📝 自测清单

- [ ] 能从一个业务目标推导出北极星 → 一级/二级 → 过程指标树
- [ ] 能用 OSM 模板回答「怎么评估某功能」类问题
- [ ] 能背出 AARRR 五阶段及每阶段代表指标
- [ ] 能设计一张指标字典并解释"为什么需要口径沉淀"
- [ ] 能手写埋点质量校验（缺失/重复/命名/时间）并给出报告
- [ ] 能讲清数仓四层职责与常见追问